# End-to-End Test-Time Training for Long Context

> **Paper:** [test-time-training.github.io/e2e.pdf](https://test-time-training.github.io/e2e.pdf)  
> **Repo:** [github.com/test-time-training/e2e](https://github.com/test-time-training/e2e)  
> **arXiv:** [2512.23675](https://arxiv.org/abs/2512.23675)

---

## Overview

This notebook walks through the key ideas in **E2E-TTT** — a method that reformulates long-context language modelling as **continual learning** rather than an architecture design problem.

### The core problem

Standard transformers use **full attention** over the entire context window. This has two costs:

| Property | Full Attention | Sliding-Window Attention + TTT |
|---|---|---|
| Reads context | ✅ Everything | ✅ Everything (at training time) |
| Inference memory | O(n) KV cache | O(window) KV cache |
| Inference compute | O(n²) | O(1) per token |
| Extrapolates beyond training length | ❌ | ✅ |

### The E2E-TTT answer

1. Use a **standard Transformer with sliding-window attention** (fixed, small window) — cheap at inference.
2. Add a small **adaptive MLP** per block that **updates its own weights** at test time via next-token prediction on the input context, effectively compressing the long context into weights.
3. **Meta-train** the model so the adaptive MLP's *initial weights* are a good starting point for fast test-time adaptation (MAML-style).

This makes the method **E2E** in two senses:
- **Test-time E2E:** the TTT objective is the same as the main objective (next-token prediction)
- **Training-time E2E:** meta-learning is end-to-end differentiable through the TTT inner loop

---

## Notebook Contents

1. [Architecture Deep Dive](#1-architecture-deep-dive)
2. [Sliding-Window Attention](#2-sliding-window-attention)
3. [Dual MLP Design](#3-dual-mlp-design)
4. [Meta-Training](#4-meta-training)
5. [Test-Time Training](#5-test-time-training)
6. [Live Demo — Pure NumPy](#6-live-demo)
7. [Visualising TTT Adaptation](#7-visualising-ttt-adaptation)
8. [Production JAX Implementation](#8-production-jax-implementation)
9. [Checkpoints & Reproducing Results](#9-checkpoints--reproducing-results)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
import sys, os

# Make ttt_e2e importable from this directory
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('NumPy', np.__version__)

---
## 1. Architecture Deep Dive

### Block structure

Each transformer block in E2E-TTT contains:

```
x  ──► LayerNorm ──► SlidingWindowAttention ──► + ──► LayerNorm ──► [AdaptiveMLP + FrozenMLP] ──► +
│                                                │                                                  │
└────────────────────────────────────────────────┘                  └──────────────────────────────┘
         residual connection 1                                              residual connection 2
```

The MLP sub-layer is **split into two parallel halves**:
- **`frozen_mlp`** — standard weights, updated only during meta-training (the "prior")
- **`adaptive_mlp`** — same architecture, but **updated at test time** via SGD on the next-token loss

In the production JAX code (`ttt/model/transformer.py`) this appears as `feed_forward` (frozen) and `feed_forward_prime` (adaptive).  
Only the **suffix blocks** (last N layers) carry the adaptive MLP.

### Why split?

- The frozen MLP remembers **world knowledge** from pre-training.
- The adaptive MLP memorises **local context** by compressing the current document into its weights.
- Together they behave like a *mixture* that can interpolate between prior knowledge and in-context information.


In [ ]:
# Visual: Block diagram
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
ax.axis('off')

def box(ax, x, y, w, h, text, color='#4A90D9', fontsize=9, text_color='white'):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='#2c3e50', linewidth=1.2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=text_color, wrap=True)

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.5))

# Input
box(ax, 0.2, 2.5, 1.2, 1, 'Input\nx', '#7f8c8d')
arrow(ax, 1.4, 3.0, 1.9, 3.0)

# LN1
box(ax, 1.9, 2.5, 1.2, 1, 'LayerNorm', '#27ae60')
arrow(ax, 3.1, 3.0, 3.6, 3.0)

# SWA
box(ax, 3.6, 2.5, 1.8, 1, 'Sliding-Window\nAttention', '#2980b9')
arrow(ax, 5.4, 3.0, 5.9, 3.0)

# Add residual (+ sign)
ax.text(5.95, 3.05, '+', fontsize=16, fontweight='bold', color='#2c3e50')
arrow(ax, 6.2, 3.0, 6.7, 3.0)

# Residual line for attention
ax.annotate('', xy=(6.1, 3.0), xytext=(0.8, 3.0),
            arrowprops=dict(arrowstyle='->', color='#95a5a6', lw=1.2,
                           connectionstyle='arc3,rad=-0.4'))

# LN2
box(ax, 6.7, 2.5, 1.2, 1, 'LayerNorm', '#27ae60')
arrow(ax, 7.9, 3.0, 8.2, 3.0)

# Frozen MLP
box(ax, 8.2, 3.7, 2.0, 0.9, 'Frozen MLP\n(world knowledge)', '#8e44ad', fontsize=8)
# Adaptive MLP  
box(ax, 8.2, 2.4, 2.0, 0.9, 'Adaptive MLP\n(test-time update)', '#e67e22', fontsize=8)

# Arrows from LN2 split
ax.annotate('', xy=(8.2, 4.15), xytext=(8.05, 3.3),
            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.2))
ax.annotate('', xy=(8.2, 2.85), xytext=(8.05, 3.0),
            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.2))

# Join and +
ax.annotate('', xy=(10.4, 3.3), xytext=(10.2, 4.15),
            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.2))
ax.annotate('', xy=(10.4, 3.1), xytext=(10.2, 2.85),
            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.2))
ax.text(10.45, 3.15, '+', fontsize=16, fontweight='bold', color='#2c3e50')
arrow(ax, 10.65, 3.2, 11.1, 3.2)

# Residual for MLP
ax.annotate('', xy=(10.55, 3.2), xytext=(6.5, 3.2),
            arrowprops=dict(arrowstyle='->', color='#95a5a6', lw=1.2,
                           connectionstyle='arc3,rad=0.35'))

# Output
box(ax, 11.1, 2.7, 0.7, 0.9, 'Out', '#7f8c8d')

ax.set_title('E2E-TTT Transformer Block', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('block_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Sliding-Window Attention

Rather than attending to all past tokens (which would be O(n²) in compute), each token can only attend to the **W most recent tokens** (its local window).

The production config uses **W = 8192** tokens. This gives O(W·n) compute — linear in sequence length.

The "lost" context beyond the window is handled by the **adaptive MLP**, which compresses it into weights.


In [ ]:
def visualise_attention_mask(seq_len=16, window=5):
    """Show what full attention vs sliding-window attention can see."""
    full_mask = np.tril(np.ones((seq_len, seq_len)))

    swa_mask = np.zeros((seq_len, seq_len))
    for i in range(seq_len):
        for j in range(max(0, i - window + 1), i + 1):
            swa_mask[i, j] = 1.0

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, mask, title in zip(axes,
                                [full_mask, swa_mask],
                                [f'Full Causal Attention\n(O(n²) — sees everything)',
                                 f'Sliding-Window Attention\n(window={window}, O(n·W) — sees last {window} tokens)']):
        im = ax.imshow(mask, cmap='Blues', vmin=0, vmax=1, aspect='auto')
        ax.set_xlabel('Key position', fontsize=10)
        ax.set_ylabel('Query position', fontsize=10)
        ax.set_title(title, fontsize=10, fontweight='bold')
        for i in range(seq_len):
            for j in range(seq_len):
                if mask[i, j] > 0:
                    ax.text(j, i, '✓', ha='center', va='center', fontsize=7, color='white')

    plt.suptitle('Attention Masks: What Each Token Can See', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('attention_masks.png', dpi=150, bbox_inches='tight')
    plt.show()

visualise_attention_mask(seq_len=16, window=5)

---
## 3. Dual MLP Design

The key idea is that the MLP capacity is split:

```python
# From ttt/model/transformer.py (JAX)
class Block(eqx.Module):
    feed_forward:       SwiGLUMLP   # frozen  — world knowledge
    feed_forward_prime: SwiGLUMLP   # adaptive — local context compression

# Forward:
h = layernorm(x)
x = x + feed_forward(h) + feed_forward_prime(h)
#           ^^^^^^^^^^^     ^^^^^^^^^^^^^^^^^^^^
#           frozen output   adapts during TTT
```

Only the **suffix blocks** (last `suffix_len=3` layers in the 125M config) carry `feed_forward_prime`.  
The inner optimizer targets only `**.suffix_blocks.feed_forward_prime.**` parameters.

### Inner optimizer

The inner (TTT) optimizer is **SGD with lr=1** per the 125M config:

```yaml
# configs/experiment/125m/pretrain/pretrain-125m-e2e.yaml
training:
  train_mode: meta
  spec_inner: ["language_model.**.suffix_blocks.feed_forward_prime.**"]
  optimizer_inner:
    optimizer_type: sgd
    lr: 1
    clip_gradient: 1.0
  mini_batch_size: 1024   # tokens per TTT gradient step
```


In [ ]:
# Show what fraction of parameters are adaptive (TTT) vs frozen
from ttt_e2e import E2ETTT

configs = [
    dict(label='Tiny (demo)',  vocab_size=64,  d_model=32,  n_layers=4,  d_ff=64),
    dict(label='Small',        vocab_size=512, d_model=128, n_layers=6,  d_ff=256),
    dict(label='Medium',       vocab_size=512, d_model=256, n_layers=8,  d_ff=512),
]

print(f"{'Config':<20} {'Total params':>14} {'TTT params':>12} {'TTT %':>8}")
print('-' * 58)
for cfg in configs:
    label = cfg.pop('label')
    m = E2ETTT(**cfg, n_heads=4, max_seq_len=512, window_size=32)
    total = m.count_params()
    ttt   = m.count_ttt_params()
    print(f"{label:<20} {total:>14,} {ttt:>12,} {100*ttt/total:>7.1f}%")
    print(f"{'  TTT block indices:':<20} {str(m.ttt_block_indices):<40}")

---
## 4. Meta-Training

Meta-training (the **outer loop**) trains the model so that the adaptive MLP starts from a good initialisation — one that can adapt quickly at test time with just a few gradient steps.

### Outer vs inner loop

```
For each sequence in the training set:
  ┌─ INNER LOOP (TTT — runs at test time too) ─────────────────────────────┐
  │  Split sequence into mini-batches of 1024 tokens                       │
  │  For each mini-batch:                                                   │
  │    1. Forward through prefix blocks (frozen weights, SWA)               │
  │    2. Forward through suffix blocks (adaptive MLP)                      │
  │    3. Compute next-token prediction loss                                │
  │    4. SGD step on adaptive MLP weights only                             │
  └────────────────────────────────────────────────────────────────────────┘
  OUTER LOOP: Backprop through the entire inner loop  ←── this is the key!
              Update ALL weights (outer optimizer = AdamW)
```

The outer loss is the **average token prediction loss across all mini-batches** after each inner step has been applied — very similar to MAML but with the same objective for inner and outer.

In the JAX code (`ttt/model/loop.py` → `train_on_sequence`), the entire inner loop is differentiated through with `eqx.filter_value_and_grad`.


In [ ]:
from ttt_e2e import E2ETTT, meta_train, cross_entropy

# Create a small model for demonstration
np.random.seed(42)
model = E2ETTT(
    vocab_size=64,
    d_model=32,
    n_heads=4,
    n_layers=4,
    d_ff=64,
    max_seq_len=256,
    window_size=16,
    ttt_batch_size=16,
    ttt_lr=0.005,
)

print(f"Model: {model.count_params():,} total params, {model.count_ttt_params():,} TTT params")
print(f"TTT block indices: {model.ttt_block_indices}")

In [ ]:
# Track meta-training loss curve
meta_losses = []

def meta_train_tracked(model, n_steps=200, seq_len=64, lr=1e-3):
    V = model.vocab_size
    for step in range(n_steps):
        pattern_len = np.random.randint(3, 8)
        pattern = np.random.randint(0, V, size=pattern_len)
        repeats = seq_len // pattern_len + 1
        seq = np.tile(pattern, repeats)[:seq_len + 1].astype(np.int32)

        logits = model.forward(seq[:-1])
        loss, grad_logits = cross_entropy(logits, seq[1:])
        meta_losses.append(loss)
        model.backward(grad_logits)

        for p, g in model.all_params_and_grads():
            p -= lr * g

print("Running meta-training (200 steps)...")
t0 = time.time()
meta_train_tracked(model, n_steps=200, seq_len=64, lr=1e-3)
print(f"Done in {time.time()-t0:.1f}s")

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
window_smooth = 10
smoothed = np.convolve(meta_losses, np.ones(window_smooth)/window_smooth, mode='valid')
ax.plot(meta_losses, alpha=0.3, color='steelblue', label='Raw loss')
ax.plot(range(window_smooth-1, len(meta_losses)), smoothed,
        color='steelblue', linewidth=2, label=f'Smoothed (w={window_smooth})')
ax.set_xlabel('Meta-training step')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Meta-Training Loss Curve', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('meta_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Test-Time Training

At inference time, when the model receives a long context document:

1. **Process the context** in mini-batches (e.g. 1024 tokens)
2. For each mini-batch, compute next-token prediction loss and **update only the adaptive MLP weights** via one SGD step
3. The adapted weights now **encode a compressed representation of the context**
4. Answer queries with the adapted model (O(W) attention, not O(n²))
5. **Restore the original weights** (W₀) for the next document

### Why this works

The adaptive MLP acts like a *write head*: it memorises patterns from the context by gradient descent.  
The sliding-window attention acts like a *read head*: it attends to recent tokens normally.  
Together they form an implicit memory — long-range via MLP weights, short-range via the window.


In [ ]:
# Demonstrate TTT adaptation
# Context: a repeating pattern the model must learn to predict

pattern = np.array([7, 13, 42, 3, 19, 55, 7, 13], dtype=np.int32)
context = np.tile(pattern, 20)   # 160 tokens

print(f"Context length : {len(context)} tokens")
print(f"Pattern        : {pattern.tolist()} (repeating x{len(context)//len(pattern)})")

# Save initial state
init_state = model.save_ttt_state()

# Measure loss BEFORE TTT
logits_before = model.forward(context[:-1])
loss_before, _ = cross_entropy(logits_before, context[1:])

# TTT over multiple passes — track per-step losses
all_ttt_losses = []
for _ in range(5):   # 5 passes over the context
    all_ttt_losses.extend(model.ttt_adapt(context))

# Measure loss AFTER TTT
logits_after = model.forward(context[:-1])
loss_after, _ = cross_entropy(logits_after, context[1:])

print(f"\nLoss before TTT : {loss_before:.4f}")
print(f"Loss after TTT  : {loss_after:.4f}")
print(f"Improvement     : {100*(loss_before-loss_after)/loss_before:.1f}%")

---
## 6. Live Demo

Full end-to-end demo using the pure-NumPy implementation from `ttt_e2e.py`.  
No GPU required — runs on CPU in seconds.


In [ ]:
# Run the full built-in demo
from ttt_e2e import demo
demo()

---
## 7. Visualising TTT Adaptation


In [ ]:
# --- Experiment: How does loss evolve during TTT? ---
np.random.seed(0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Three different patterns to test adaptation speed
test_patterns = [
    ('Short pattern (3)',  np.array([1, 5, 9], dtype=np.int32)),
    ('Medium pattern (6)', np.array([7, 13, 42, 3, 19, 55], dtype=np.int32)),
    ('Long pattern (10)',  np.array([2, 17, 4, 63, 9, 31, 7, 44, 11, 58], dtype=np.int32)),
]

for ax, (name, pattern) in zip(axes, test_patterns):
    context = np.tile(pattern, 30)[:128]

    # Fresh model for each test
    m = E2ETTT(vocab_size=64, d_model=32, n_heads=4, n_layers=4,
               d_ff=64, max_seq_len=256, window_size=16,
               ttt_batch_size=16, ttt_lr=0.01)
    # Quick meta-train to get a reasonable init
    meta_train(m, n_steps=100, seq_len=48, lr=1e-3)

    init_state = m.save_ttt_state()

    # TTT with tracked losses
    ttt_losses_exp = []
    for _ in range(6):
        ttt_losses_exp.extend(m.ttt_adapt(context))

    ax.plot(ttt_losses_exp, color='#e67e22', linewidth=2)
    ax.fill_between(range(len(ttt_losses_exp)), ttt_losses_exp, alpha=0.15, color='#e67e22')
    ax.set_title(name, fontweight='bold', fontsize=10)
    ax.set_xlabel('TTT step')
    ax.set_ylabel('NTP loss' if ax == axes[0] else '')
    ax.set_ylim(bottom=0)

plt.suptitle('TTT Loss Curves for Different Pattern Complexities', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ttt_adaptation_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Experiment: TTT vs no TTT prediction accuracy ---
np.random.seed(42)

np.random.seed(42)
m = E2ETTT(vocab_size=64, d_model=32, n_heads=4, n_layers=4,
           d_ff=64, max_seq_len=256, window_size=16,
           ttt_batch_size=16, ttt_lr=0.005)
meta_train(m, n_steps=200, seq_len=64, lr=1e-3)

pattern = np.array([7, 13, 42, 3, 19, 55, 7, 13], dtype=np.int32)
context = np.tile(pattern, 24)  # 192 tokens

# Without TTT
logits_no_ttt = m.forward(context[:-1])
preds_no_ttt  = np.argmax(logits_no_ttt, axis=-1)

# With TTT
init_state = m.save_ttt_state()
for _ in range(3):
    m.ttt_adapt(context)
logits_ttt = m.forward(context[:-1])
preds_ttt  = np.argmax(logits_ttt, axis=-1)
m.restore_ttt_state(init_state)

targets = context[1:]

# Token-level accuracy
acc_no_ttt = np.cumsum(preds_no_ttt == targets) / np.arange(1, len(targets)+1)
acc_ttt    = np.cumsum(preds_ttt    == targets) / np.arange(1, len(targets)+1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Accuracy over position
ax = axes[0]
ax.plot(acc_no_ttt, label='No TTT', color='steelblue', linewidth=1.5)
ax.plot(acc_ttt,    label='With TTT', color='#e67e22', linewidth=2)
ax.set_xlabel('Token position')
ax.set_ylabel('Cumulative accuracy')
ax.set_title('Cumulative Token Prediction Accuracy', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)

# First 64 tokens side-by-side
ax = axes[1]
N = 48
x = np.arange(N)
correct_no_ttt = (preds_no_ttt[:N] == targets[:N]).astype(float)
correct_ttt    = (preds_ttt[:N]    == targets[:N]).astype(float)
ax.bar(x - 0.2, correct_no_ttt, width=0.35, label='No TTT', color='steelblue', alpha=0.7)
ax.bar(x + 0.2, correct_ttt,    width=0.35, label='With TTT', color='#e67e22', alpha=0.9)
ax.set_xlabel('Token position (first 48)')
ax.set_ylabel('Correct prediction (1=yes)')
ax.set_title('Per-Token Correctness (first 48 tokens)', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('ttt_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Overall accuracy — No TTT: {np.mean(preds_no_ttt==targets):.1%}  |  With TTT: {np.mean(preds_ttt==targets):.1%}")

---
## 8. Production JAX Implementation

The `test-time-training/e2e` repo is the full production implementation. Here's how the key components map to the ideas above.

### Repository layout

```
e2e/
├── ttt/
│   ├── model/
│   │   ├── transformer.py   # SwiGLUMLP, Block, BlockCollectionSplit, MetaModel, CausalLM
│   │   ├── attention.py     # SWA (sliding-window attention) with RoPE
│   │   ├── loop.py          # train_on_sequence, Evaluator
│   │   └── loss.py          # cross_entropy_loss_and_accuracy
│   ├── config.py            # Hydra config dataclasses
│   ├── train.py             # Main training entry point
│   └── optimizers.py        # make_optimizer (SGD inner, AdamW outer)
└── configs/
    ├── model/               # 125m / 350m / 760m / 1b / 3b
    ├── experiment/          # pretrain + extension configs
    └── deploy/              # interactive (single node) + submitit (Slurm)
```

### Key design choices in the JAX code

| Component | Implementation |
|---|---|
| Attention | `SWA` — sliding-window with RoPE (θ=500000) |
| MLP | `SwiGLUMLP` — 3-matrix SwiGLU (w1, w2, w3) |
| Norm | `RMSNorm` (pre-norm + post-norm for stability) |
| Inner optimizer | SGD, lr=1, gradient clip=1.0 |
| Outer optimizer | AdamW |
| Parallelism | `filter_vmap` over data-parallel devices |
| Checkpointing | Activation remat via `maybe_double_remat` |
| Meta-gradient | `eqx.filter_value_and_grad` through the inner loop |

### `MetaModel.loss_for_sequence` — the heart of the code

```python
# Simplified pseudocode of the meta-training inner loop
def loss_for_sequence(self, seq, state):
    # Split model into prefix (frozen SWA) and suffix (adaptive MLP) blocks
    prefix_blocks, suffix_blocks = split_at_suffix_len(blocks, suffix_len=3)

    # 1. Run prefix blocks ONCE — cheap, no gradients needed here
    prefix_output = prefix_call(prefix_blocks, embed(seq))

    # 2. Inner loop over mini-batches of the suffix
    for chunk in chunks(seq, size=mini_batch_size):   # 1024 tokens each
        loss = suffix_call(suffix_blocks, prefix_output[chunk])
        grads = grad(loss, wrt=feed_forward_prime)
        suffix_blocks = sgd_update(suffix_blocks, grads, lr=1.0)

    # 3. Outer loss = average loss across all chunks (after each inner step)
    return average_chunk_loss
```


---
## 9. Checkpoints & Reproducing Results

### Available checkpoints (Google Cloud Storage)

| Model | Dataset | Context | Bucket |
|---|---|---|---|
| 125M TTT-E2E (1× Chinchilla) | DCLM | 8K | `gs://ttt-e2e-checkpoints/125m_ttt_e2e_pretrain_dclm_8k_1x_cc` |
| 1B TTT-E2E (1× Chinchilla) | DCLM | 8K | `gs://ttt-e2e-checkpoints/1b_ttt_e2e_pretrain_dclm_8k_1x_cc` |
| 3B TTT-E2E (3× Chinchilla) | DCLM | 8K | `gs://ttt-e2e-checkpoints/3b_ttt_e2e_pretrain_dclm_8k_3x_cc` |
| 125M TTT-E2E (1× Chinchilla) | Books | 8K | `gs://ttt-e2e-checkpoints/125m_ttt_e2e_finetune_books_8k_1x_cc` |
| 1B TTT-E2E (1× Chinchilla) | Books | 8K | `gs://ttt-e2e-checkpoints/1b_ttt_e2e_finetune_books_8k_1x_cc` |
| 3B TTT-E2E (3× Chinchilla) | Books | 8K | `gs://ttt-e2e-checkpoints/3b_ttt_e2e_finetune_books_8k_3x_cc` |
| 3B TTT-E2E (3× Chinchilla) | Books | **128K** | `gs://ttt-e2e-checkpoints/3b_ttt_e2e_finetune_books_128k_3x_cc` |

> **Note:** These buckets have **Requester Pays** enabled. You need a GCP project to download.

### Setup & run

```bash
# 1. Install uv
curl -LsSf https://astral.sh/uv/install.sh | sh

# 2. Download a tokenised dataset
gcloud storage cp -r gs://llama3-dclm-filter-8k/ llama3-dclm-filter-8k

# 3. Fill in deploy_paths in configs/deploy/interactive.yaml

# 4. Run 125M pre-training (single interactive node)
uv run --exact train \
  +deploy=interactive \
  +experiment=125m/pretrain/pretrain-125m-e2e \
  training.wandb_entity=MY_ENTITY \
  training.wandb_project=MY_PROJECT \
  training.wandb_key=MY_KEY

# 5. Load from checkpoint for extension to 32K context
uv run --exact train \
  +deploy=interactive \
  +experiment=125m/extension/ext-125m-e2e-32K \
  training.resume_exp_name=pretrain-125m-e2e \
  training.load_part=params \
  training.wandb_entity=MY_ENTITY \
  training.wandb_project=MY_PROJECT \
  training.wandb_key=MY_KEY
```

### System requirements

- CUDA 12.8.1, cuDNN 9.8.0, NCCL 2.26.2
- JAX 0.4.x + Equinox + Optax
- Weights & Biases account (for logging)


---
## Summary

| Concept | What it does | Where in code |
|---|---|---|
| **Sliding-Window Attention** | O(W) per-token cost; limits context to last W tokens | `ttt/model/attention.py` → `SWA` |
| **Adaptive MLP (prime)** | Compressed memory — updated by TTT | `feed_forward_prime` in `Block` |
| **Frozen MLP** | World knowledge from pre-training | `feed_forward` in `Block` |
| **Prefix blocks** | First N-3 layers — run once, no TTT | `BlockCollectionSplit.prefix_call` |
| **Suffix blocks** | Last 3 layers — carry adaptive MLP, run per mini-batch | `BlockCollectionSplit.suffix_call` |
| **Inner loop** | SGD on adaptive MLP weights (lr=1, per 1024-token chunk) | `MetaModel.inner_loop_step` |
| **Outer loop** | AdamW on ALL weights via BPTT through inner loop | `train_on_sequence` → `filter_value_and_grad` |
| **Weight restore** | Reset adaptive MLP to W₀ between documents | `save_ttt_state` / `restore_ttt_state` |

### Key result from the paper

E2E-TTT achieves **better perplexity than full-attention baselines on long documents** (Books, 32K–128K context) while using **O(1) inference compute** per token instead of O(n²). The meta-learned initialisation is critical — random initialisation of the adaptive MLP performs much worse.

---
*Paper: [test-time-training.github.io/e2e.pdf](https://test-time-training.github.io/e2e.pdf)*  
*Code: [github.com/test-time-training/e2e](https://github.com/test-time-training/e2e)*
